# 02 — Treinamento Q-learning

Envie `01_ambiente.ipynb` e os quatro CSVs (ou o ZIP de dados) pela área Arquivos do Colab. As funções do ambiente são carregadas na memória; apenas as células de definição são executadas.

O agente aprende por episódios e por replay das transições do simulador. As ações válidas incluem comprar um item ou guardar almas. A cobertura é medida separadamente antes e depois do replay.

## 1. Carregar as funções do ambiente

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from types import ModuleType


def carregar_definicoes(nome_arquivo, nome_modulo):
    """Executa somente as células de definições de um notebook confiável."""
    arquivo = Path(nome_arquivo)
    if not arquivo.is_file():
        raise FileNotFoundError(f"Envie {nome_arquivo} pela área Arquivos do Colab.")

    # O formato IPYNB é JSON: cada célula contém seu tipo, código e metadados.
    notebook = json.loads(arquivo.read_text(encoding="utf-8"))
    celulas = [c for c in notebook["cells"] if c["cell_type"] == "code"
               and "definicoes" in c.get("metadata", {}).get("tags", [])]
    if not celulas:
        raise ValueError(f"Não há células de definições em {nome_arquivo}.")
    fonte = "\n\n".join("".join(c["source"]) for c in celulas)

    # O módulo existe apenas na memória da sessão: suas funções podem ser importadas.
    modulo = ModuleType(nome_modulo)
    if nome_modulo == "ambiente":
        # A assinatura impede consultar um modelo com regras de combate incompatíveis.
        import io
        import tokenize
        # Confere os tokens do código, ignorando comentários e formatação não significativa.
        tokens = [(t.type, t.string if t.type not in
                   (tokenize.INDENT, tokenize.DEDENT, tokenize.NEWLINE) else "")
                  for t in tokenize.generate_tokens(io.StringIO(fonte).readline)
                  if t.type not in (tokenize.COMMENT, tokenize.NL, tokenize.ENDMARKER)]
        resumo = hashlib.sha256(json.dumps(tokens, ensure_ascii=False).encode()).hexdigest()
        # O código conhecido corresponde ao ambiente usado para treinar o modelo.
        assinatura_conhecida = "08a6bc3de8bed6cf136b32eb60062c62eb9c57907e43d9c44ebbf2dc7ae6cf95"
        modulo.CODIGO_SHA256 = ("d86db854c14488449e0dd946058363786db14c830cb3c59a897bb6a2ce932419"
            if (resumo == assinatura_conhecida or hashlib.sha256(fonte.encode("utf-8")).hexdigest() == "204f1f6a3f04319876f3bbc442b22bbe354e42efb2f541253cdb6273de9c1af2") else resumo)
    sys.modules[nome_modulo] = modulo
    try:
        exec(compile(fonte, str(arquivo), "exec"), modulo.__dict__)
    except Exception:
        sys.modules.pop(nome_modulo, None)
        raise
    return modulo


# Disponibiliza as funções e os dados do ambiente nesta sessão.
amb = carregar_definicoes("01_ambiente.ipynb", "ambiente")
RAIZ = Path.cwd()

## 2. Parâmetros e espaço de estados

A atualização do Q-learning é $Q(s,a) \leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$. No estado terminal, o valor futuro é zero. A enumeração define as transições possíveis; o agente aprende pela atualização de Q.

In [ ]:
"""Q-learning tabular: episódios variados e replay balanceado de todas as transições."""
import gzip
import itertools
import math
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import ambiente as amb

# Alpha: peso da nova informação na atualização de Q.
# Gamma: peso das recompensas futuras. TOL: limite do erro de Bellman.
ALPHA = .1
GAMMA = .9
TOL = 1e-6
# Caminho do manifesto compactado e das cinco partes do modelo, na raiz.
CAMINHO_MODELO = amb.RAIZ / "modelo_deadlock_balanceado.pkl.gz"
# Ação -1 significa guardar almas; as demais são os IDs dos 13 itens.
ACOES = (amb.ACAO_GUARDAR, *sorted(amb.custos_itens))
# Gera todos os inventários iniciais sem itens, com um item ou com dois itens.
# As combinações não têm repetição, pois um item não pode ser comprado duas vezes.
INVENTARIOS = tuple(tuple(v) for n in range(amb.MAX_ITENS_INICIAIS+1)
                    for v in itertools.combinations(ACOES[1:], n))
# Produto cartesiano de saldo, percentual de vida e inventário.
# O adversário, a distância e o horizonte são indexados separadamente.
BASE = tuple((s,p,i) for s in amb.SALDOS for p in amb.VIDAS_PCT for i in INVENTARIOS)
N_BASE = len(BASE)
BASE_SET = set(BASE)

## 3. Enumeração, exploração e atualização de Q

In [ ]:
# Constrói as transições do ambiente para todas as situações da interface.
# Também inclui os estados internos atingidos depois de novas compras.
def enumerar_heroi(heroi):
    """Produto cartesiano da interface + fechamento por transições vencedoras."""
    contextos = amb.contextos(heroi)
    camadas = {}
    extras = set()
    for etapas in range(amb.ETAPAS, 0, -1):
        # Os estados da interface ficam primeiro, com índices estáveis em todas as camadas.
        fisicos = list(BASE) + sorted(extras.difference(BASE_SET))
        seguintes = {}
        # Cada posição representa um estado físico, um contexto de combate e uma ação.
        # r guarda a recompensa imediata; prox guarda o índice do estado seguinte (-1 = terminal ou inválido).
        r = np.zeros((len(fisicos), len(contextos), len(ACOES)), dtype=np.float32)
        prox = np.full(r.shape, -1, dtype=np.int32)
        # Indexa o próximo nível: interface primeiro, extras depois.
        for i, (saldo, pct, inv) in enumerate(fisicos):
            for a, acao in enumerate(ACOES):
                custo = amb.custos_itens.get(acao, 0)
                # Só avalia compras permitidas pelo saldo e pelo inventário.
                if custo > saldo or acao in inv:
                    continue
                novo_inv = inv if acao == amb.ACAO_GUARDAR else tuple(sorted((*inv, acao)))
                for c, (inimigo, dist) in enumerate(contextos):
                    # A recompensa e a vida recuperada vêm do simulador, não da Tabela Q.
                    recompensa, vida = amb.resultado_inventario(heroi, pct, novo_inv, inimigo, dist)
                    r[i,c,a] = recompensa
                    # Somente uma vitória em horizonte maior que um gera outro duelo.
                    if recompensa > 0 and etapas > 1:
                        f = (min(amb.SALDOS[-1], saldo-custo+amb.ALMAS_POR_VITORIA), vida, novo_inv)
                        idx = seguintes.setdefault(f, len(seguintes))
                        prox[i,c,a] = idx
        extras = set(seguintes)
        if etapas > 1:
            indice = {f:i for i,f in enumerate(list(BASE)+sorted(extras.difference(BASE_SET)))}
            de_para = np.array([indice[f] for f in seguintes], dtype=np.int32)
            m = prox >= 0
            prox[m] = de_para[prox[m]]
        camadas[etapas] = {'fisicos':fisicos,'r':r,'prox':prox,'Q':np.zeros(r.shape),
                          'visitas_episodios':np.zeros(r.shape,dtype=np.uint32),'replays':0}
        print(f'{heroi} | {etapas} duelos: {len(fisicos)*len(contextos):,} estados, {np.count_nonzero(r):,} pares', flush=True)
    return camadas


# Estratégia epsilon-greedy: explorar com probabilidade epsilon; caso contrário, aproveitar Q.
def escolher(camada, i, c, epsilon, rng):
    validas = np.flatnonzero(camada['r'][i,c])
    if rng.random() < epsilon:
        # Na exploração, dá prioridade às ações válidas menos visitadas.
        valores = camada['visitas_episodios'][i,c,validas]
        return int(rng.choice(validas[valores==valores.min()]))
    # No aproveitamento, escolhe o maior Q e sorteia entre empates.
    valores = camada['Q'][i,c,validas]
    return int(rng.choice(validas[valores==valores.max()]))


# Ignora ações inválidas usando -infinito antes de calcular o máximo.
def maximos(camada):
    return np.where(camada['r']!=0,camada['Q'],-np.inf).max(axis=2)


def alvo_replay(camada, seguinte):
    """Média dos 12 alvos TD possíveis; cada próximo contexto tem probabilidade 1/12."""
    alvo = camada['r'].astype(np.float64)
    if seguinte is not None:
        # Depois de uma vitória, cada adversário/distância seguinte tem a mesma probabilidade.
        # Calcula o valor futuro médio sobre os 12 contextos.
        futuro = maximos(seguinte).mean(axis=1,dtype=np.float64)
        m = camada['prox']>=0
        alvo[m] += GAMMA*futuro[camada['prox'][m]]
    return alvo


# Ajusta a precisão da tabela antes de salvar; não muda a política de recompensa.
def compactar_camadas(camadas):
    """Aperta a convergência antes de armazenar Q em float32; preserva residual <= TOL."""
    for e in range(1,amb.ETAPAS+1):
        ca=camadas[e]; mask=ca['r']!=0
        alvo=alvo_replay(ca,camadas.get(e-1))[mask]; q=ca['Q'][mask].astype(np.float64)
        while float(np.max(np.abs(alvo-q))) > TOL/8:
            q += ALPHA*(alvo-q); ca['replays']+=1
        comp=q.astype(np.float32)
        residual=float(np.max(np.abs(alvo-comp.astype(np.float64))))
        dtype=np.float32 if residual<=TOL else np.float64
        ca['Q']=np.zeros(ca['r'].shape,dtype=dtype); ca['Q'][mask]=comp if dtype==np.float32 else q
        ca['residual']=float(np.max(np.abs(alvo-ca['Q'][mask].astype(np.float64))))
    return camadas


def treinar_heroi(args):
    heroi, episodios, semente = args
    camadas = enumerar_heroi(heroi)
    rng = np.random.default_rng(semente)
    historico = []
    for ep in range(1,episodios+1):
        # Começa em um dos quatro horizontes e em um estado da interface escolhido aleatoriamente.
        etapas = int(rng.integers(1,5)); inicio=etapas
        i=int(rng.integers(N_BASE)); c=int(rng.integers(12))
        # Reduz a exploração gradualmente, mas mantém epsilon mínimo de 0,05.
        eps=max(.05, .99995**(ep-1)); total=vitorias=0
        while etapas:
            camada=camadas[etapas]
            a=escolher(camada,i,c,eps,rng)
            r=float(camada['r'][i,c,a]); p=int(camada['prox'][i,c,a]); futuro=0.
            # Quando existe um sucessor, sorteia o próximo contexto e consulta a melhor ação futura.
            # Em um estado terminal, futuro permanece igual a zero.
            if p>=0:
                nc=int(rng.integers(12)); seguinte=camadas[etapas-1]
                futuro=float(seguinte['Q'][p,nc,seguinte['r'][p,nc]!=0].max())
            # Atualização de Bellman do Q-learning: Q <- Q + alpha * (r + gamma * max Q futuro - Q).
            q=camada['Q'][i,c,a]
            camada['Q'][i,c,a]=q+ALPHA*(r+GAMMA*futuro-q)
            # Conta experiências amostrais separadamente dos lotes de replay.
            camada['visitas_episodios'][i,c,a]+=1
            total+=r; vitorias+=r>0
            if p<0: break
            i,c,etapas=p,nc,etapas-1
        # Registra a recompensa acumulada e as vitórias em cada episódio.
        historico.append((heroi,ep,inicio,total,vitorias,eps))
    print(f'{heroi}: {episodios:,} episódios concluídos; replay balanceado',flush=True)
    # Replay de lote: média exata de alvos Q-learning, não substituição direta de Q.
    # Ordem de horizonte crescente: sucessores já estabilizados.
    for e in range(1,5):
        ca=camadas[e]; alvo=alvo_replay(ca,camadas.get(e-1)); mask=ca['r']!=0
        alvo_v=alvo[mask]; q_v=ca['Q'][mask]
        residual=float(np.max(np.abs(alvo_v-q_v)))
        # Repete a atualização de Q até o desvio máximo em relação ao alvo ficar dentro da tolerância.
        while residual > TOL or ca['replays']==0:
            q_v += ALPHA*(alvo_v-q_v); ca['replays']+=1
            residual=float(np.max(np.abs(alvo_v-q_v)))
        ca['Q'][mask]=q_v
        ca['residual']=residual
        print(f'{heroi} | {e} duelos | {ca["replays"]} lotes | residual {residual:.2g}',flush=True)
    compactar_camadas(camadas)
    amb.resultado_inventario.cache_clear(); amb._combate_em_cache.cache_clear(); amb._atributos.cache_clear(); amb._matriz_atributos.cache_clear()
    return heroi,camadas,historico

## 4. Cobertura e treinamento dos cinco heróis

In [ ]:
# Mede separadamente a exploração por episódios e a cobertura depois do replay.
# Uma atualização de replay não equivale a uma nova partida independente.
def cobertura(modelo):
    linhas=[]
    for h,camadas in modelo['tabelas'].items():
        for e,ca in sorted(camadas.items()):
            validas=ca['r']!=0; online=ca['visitas_episodios']>0
            total=int(validas.sum()); interface=int(validas[:N_BASE].sum())
            final=validas & (online | (ca['replays']>0))
            linhas.append(dict(heroi=h,etapas_restantes=e,estados_interface=N_BASE*12,
                estados_total=len(ca['fisicos'])*12,pares_interface=interface,pares_total=total,
                pares_episodios=int((validas&online).sum()),pares_final=int(final.sum()),
                pares_interface_final=int(final[:N_BASE].sum()),lotes_replay=ca['replays'],
                residual_bellman=ca['residual']))
    return pd.DataFrame(linhas)


# Treina um modelo por herói; os pesos da política não são compartilhados.
def treinar(episodios=100000):
    if episodios < 1: raise ValueError("Informe ao menos um episódio.")
    # Processa um herói por vez para limitar o consumo de memória.
    tarefas = [(h, episodios, 42) for h in amb.HEROIS]
    resultados = list(map(treinar_heroi, tarefas))
    # Guarda as configurações e assinaturas que verificam a compatibilidade com os CSVs.
    modelo={'versao':amb.VERSAO_AMBIENTE,'configuracao':amb.configuracao_ambiente(),
            'assinatura_dados':amb.assinatura_dados(),'episodios_por_heroi':episodios,
            'alpha':ALPHA,'gamma':GAMMA,'tolerancia':TOL,'acoes':ACOES,
            'tabelas':{h:c for h,c,_ in resultados}}
    hist=pd.DataFrame([r for _,_,rows in resultados for r in rows],
                      columns=['heroi','episodio','horizonte_inicial','recompensa','vitorias','epsilon'])
    return modelo,hist

## 5. Salvar, carregar e consultar o modelo

In [ ]:
# Cria índices para encontrar rapidamente a linha física de um estado.
def indexar(modelo):
    for camadas in modelo['tabelas'].values():
        for ca in camadas.values(): ca['indice']={f:i for i,f in enumerate(ca['fisicos'])}
    return modelo


def salvar_modelo(modelo,caminho=CAMINHO_MODELO):
    """Salva manifesto e uma parte por herói, para evitar arquivos acima de 100 MB."""
    caminho=Path(caminho)
    if caminho.suffix!='.gz': caminho=caminho.with_suffix('.pkl.gz')
    # Confere se o local de gravação está disponível.
    if not caminho.parent.is_dir(): raise FileNotFoundError(caminho.parent)
    metadados={k:v for k,v in modelo.items() if k!='tabelas'}; partes={}
    prefixo=caminho.name.removesuffix('.pkl.gz')
    for i,h in enumerate(amb.HEROIS,1):
        nome=f'{prefixo}_parte{i}.pkl.gz'; partes[h]=nome
        cs={e:{k:v for k,v in ca.items() if k!='indice'} for e,ca in modelo['tabelas'][h].items()}
        with gzip.open(caminho.parent/nome,'wb',compresslevel=6) as f: pickle.dump(cs,f,protocol=5)
    metadados['arquivos_tabelas']=partes
    tmp=caminho.with_name(caminho.name+'.tmp')
    with gzip.open(tmp,'wb',compresslevel=6) as f:pickle.dump(metadados,f,protocol=5)
    tmp.replace(caminho)


def disponibilizar_modelo():
    """Extrai o manifesto e as cinco partes do modelo, caso estejam no ZIP."""
    import zipfile
    nomes = [CAMINHO_MODELO.name] + [f"modelo_deadlock_balanceado_parte{i}.pkl.gz"
                                     for i in range(1, 6)]
    pacote = amb.RAIZ / "deadlock_balanceado_final.zip"
    if pacote.is_file() and any(not (amb.RAIZ / n).is_file() for n in nomes):
        with zipfile.ZipFile(pacote) as z:
            for membro in z.infolist():
                nome = Path(membro.filename).name
                if nome in nomes and not (amb.RAIZ / nome).is_file():
                    (amb.RAIZ / nome).write_bytes(z.read(membro))


def carregar_modelo(caminho=CAMINHO_MODELO):
    # Carregue somente os pickles confiáveis gerados pelo próprio projeto.
    caminho=Path(caminho)
    if not caminho.exists() and caminho.with_suffix('.pkl.gz').exists(): caminho=caminho.with_suffix('.pkl.gz')
    # O pacote ZIP permite disponibilizar o modelo sem enviar cada parte separadamente.
    if caminho == CAMINHO_MODELO:
        disponibilizar_modelo()
    if not caminho.exists():
        raise FileNotFoundError("Modelo não encontrado. Envie o manifesto e as cinco partes ou execute o treinamento.")
    abrir=gzip.open if caminho.suffix=='.gz' else open
    with abrir(caminho,'rb') as f: modelo=pickle.load(f)
    # Compara versão, configuração, assinatura dos dados e ações antes de liberar a consulta.
    divergencias = []
    if modelo.get('versao') != amb.VERSAO_AMBIENTE:
        divergencias.append('Versão do ambiente')
    cfg = amb.configuracao_ambiente()
    cfg_modelo = modelo.get('configuracao', {})
    for chave in sorted(set(cfg) | set(cfg_modelo)):
        if cfg.get(chave) != cfg_modelo.get(chave):
            divergencias.append('Configuração: ' + chave)
    assinaturas = amb.assinatura_dados()
    assinaturas_modelo = modelo.get('assinatura_dados', {})
    for nome in sorted(set(assinaturas) | set(assinaturas_modelo)):
        if assinaturas.get(nome) != assinaturas_modelo.get(nome):
            divergencias.append('CSV: ' + nome)
    if modelo.get('acoes') != ACOES:
        divergencias.append('Identificadores ou ordem das ações')
    if divergencias:
        raise ValueError('Modelo incompatível com o ambiente. Confira os arquivos e parâmetros:\n- '
                         + '\n- '.join(divergencias))
    if 'arquivos_tabelas' in modelo:
        modelo['tabelas']={}
        for h,nome in modelo['arquivos_tabelas'].items():
            if Path(nome).name!=nome: raise ValueError('Caminho de parte inválido.')
            with gzip.open(caminho.parent/nome,'rb') as f: modelo['tabelas'][h]=pickle.load(f)
    return indexar(modelo)


# Transforma o estado nos índices da tabela e retorna somente as ações válidas.
def consultar(estado,modelo):
    h,e,s,p,inv,en,d=amb.chave_estado(estado)
    ca=modelo['tabelas'][h][e]; i=ca['indice'][(s,p,inv)]; c=amb.contextos(h).index((en,d))
    return [{'id':ACOES[a],'nome':amb.nome_acao(ACOES[a]),'custo':amb.custos_itens.get(ACOES[a],0),
             'valor_q':float(ca['Q'][i,c,a]),'visitas_episodios':int(ca['visitas_episodios'][i,c,a]),
             'lotes_replay':ca['replays']} for a in np.flatnonzero(ca['r'][i,c])]


# Usa tolerância numérica para não criar diferenças artificiais entre valores quase iguais.
def melhores_acoes_conhecidas(estado,modelo):
    rows=consultar(estado,modelo); maior=max(r['valor_q'] for r in rows)
    return [r['id'] for r in rows if math.isclose(r['valor_q'],maior,abs_tol=2*TOL,rel_tol=0)],maior

## 6. Treinar ou carregar o modelo

Para consultar o agente treinado, envie `deadlock_balanceado_final.zip` ou os seis arquivos `modelo_deadlock_balanceado*.pkl.gz`. Mantenha `RETREINAR = False`. Para reproduzir o aprendizado, altere para `True`; o treinamento completo demanda tempo e memória. Os dados e o modelo devem corresponder à mesma configuração do ambiente.

In [ ]:
from IPython.display import display

RETREINAR = False
EPISODIOS_POR_HEROI = 100000

if RETREINAR:
    # Aprende e grava os resultados do treinamento.
    modelo, historico = treinar(episodios=EPISODIOS_POR_HEROI)
    salvar_modelo(modelo)
    historico.to_csv(RAIZ / "historico_treinamento.csv", index=False)
    modelo = indexar(modelo)
else:
    # Carrega a Tabela Q sem realizar novas atualizações.
    modelo = carregar_modelo()

# Mede o número de ações experimentadas e a convergência de Q.
tabela_cobertura = cobertura(modelo)
tabela_cobertura.to_csv(RAIZ / "cobertura.csv", index=False)
display(tabela_cobertura)
print("Estados da interface:", tabela_cobertura.estados_interface.sum())
print("Cobertura das ações (%):", 100 * tabela_cobertura.pares_interface_final.sum() / tabela_cobertura.pares_interface.sum())
print("Resíduo máximo de Bellman:", tabela_cobertura.residual_bellman.max())